## 03. 지도 프로토타입

서울 강남구 매매 데이터로 V-World 지도 + 랭킹 원 프로토타입을 테스트합니다.

In [ ]:
import sys
from pathlib import Path

# 프로젝트 루트를 sys.path에 추가
PROJECT_ROOT = str(Path.cwd().resolve().parents[2])
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from projects.part09_trade_map.dashboard.pivot import generate_pivot_table
from projects.part09_trade_map.dashboard.map_builder import create_ranking_map

### 1. 피벗테이블 생성 (서울 강남구 2024년)

In [ ]:
df = generate_pivot_table(
    거래유형='매매',
    시도='서울특별시',
    시군구='강남구',
    연도_시작=2024,
    연도_끝=2024,
)
print(f"결과 행 수: {len(df)}")
print(f"좌표 매칭: {df['경도'].notna().sum()}/{len(df)}")
print(f"세대수 매칭: {df['세대수'].notna().sum()}/{len(df)}")
display(df.head(20))

### 2. 거래량 테이블 (계약연도별)

In [ ]:
from projects.part09_trade_map.dashboard.preprocessing import get_year_range

# 전체 연도 범위로 데이터 조회
min_year, max_year = get_year_range('매매')
print(f'매매 데이터 연도 범위: {min_year}~{max_year}년')

df_all = generate_pivot_table(
    거래유형='매매',
    시도='서울특별시',
    시군구='강남구',
    연도_시작=min_year,
    연도_끝=max_year,
)
print(f'전체 연도 결과: {len(df_all)}건, 연도: {sorted(df_all["계약연도"].unique())}')

# 계약연도를 컬럼으로 펼친 거래량 피벗 테이블
index_cols = ['시도', '시군구', '읍면동', '단지명', '건축년도', '연식_구분', '전용면적_구분', '세대수', '경도', '위도']

거래량_table = df_all.pivot_table(
    index=index_cols,
    columns='계약연도',
    values='거래량',
    aggfunc='sum',
).reset_index()

거래량_table.columns.name = None

print(f'\n거래량 테이블: {len(거래량_table)}건')
display(거래량_table.head(20))

### 3. 회전율 테이블 (계약연도별)

In [ ]:
# 계약연도를 컬럼으로 펼친 회전율 피벗 테이블
회전율_table = df_all.pivot_table(
    index=index_cols,
    columns='계약연도',
    values='회전율',
    aggfunc='sum',
).reset_index()

회전율_table.columns.name = None

print(f'회전율 테이블: {len(회전율_table)}건')
display(회전율_table.head(20))

### 4. 랭킹 지도 생성 (거래량 기준)

In [ ]:
# 거래량 기준 상위 20개 랭킹 지도
m = create_ranking_map(df, 랭킹기준='거래량', 랭킹개수=20)
m

### 5. 회전율 기준 지도

In [ ]:
# 회전율 기준 상위 20개 랭킹 지도
m2 = create_ranking_map(df, 랭킹기준='회전율', 랭킹개수=20)
m2

### 6. HTML로 저장 (공유용)

In [ ]:
# 지도를 HTML 파일로 저장
m.save('ranking_map_gangnam_2024.html')
print('지도가 ranking_map_gangnam_2024.html로 저장되었습니다.')